### Библиотеки

In [8]:
from IPython.display import Video
import cv2
import wget
from tqdm import tqdm
import glob
import numpy as np
import pandas as pd
import yaml
from datetime import datetime
from pathlib import Path
from typing import List

import av
import numpy as np
import torch
import torchvision
from diffusers import AutoencoderKL, DDIMScheduler
from diffusers.pipelines.stable_diffusion import StableDiffusionPipeline
from einops import repeat
from omegaconf import OmegaConf
from PIL import Image
from torchvision import transforms
from transformers import CLIPVisionModelWithProjection

import sys
sys.path.append("../modules")
sys.path.append("../dataset")
sys.path.append("../Moore-AnimateAnyone")

from src.dwpose import DWposeDetector
from src.utils.util import get_fps, read_frames, save_videos_from_pil

from src.models.pose_guider import PoseGuider
from src.models.unet_2d_condition import UNet2DConditionModel
from src.models.unet_3d import UNet3DConditionModel
from src.pipelines.pipeline_pose2vid_long import Pose2VideoPipeline
from src.utils.util import get_fps, read_frames, save_videos_grid

from preprocessing import save_pose_from_mp4, generate_vid_triplets

### Разметка видео для позы

In [10]:
import os
os.chdir('../Moore-AnimateAnyone')
from src.dwpose import DWposeDetector

detector = DWposeDetector()
detector = detector.to(f"cuda")
os.chdir('../notebooks')

2025-05-12 12:09:38.593244147 [W:onnxruntime:, session_state.cc:1162 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-05-12 12:09:38.593273626 [W:onnxruntime:, session_state.cc:1164 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [11]:
np.random.seed(42)
vid_path_list = glob.glob('../dataset/test/fps_10/*')
save_dir_path = '../dataset/test/inference/'

generate_vid_triplets(vid_path_list, save_dir_path, detector, n_examples=10, clip_length=90)

generation: 9: 100%|██████████| 90/90 [00:02<00:00, 34.57it/s]


In [5]:
# для одного файла по конкретному видео
# file_path = "../dataset/test/stage_2/sample_0/orig_vid.mp4"
# out_path = "../dataset/test/stage_2/sample_0/pose_vid.mp4"
# save_pose_from_mp4(file_path, out_path, detector)

### Конфигурация конфига

In [12]:
# список файлов и видео поз для генерации
list_files = []

for i in range(10):
    pose = f"../dataset/test/inference/sample_{i}/pose_vid.mp4"
    ref = f"../dataset/test/inference/sample_{i}/ref_image.png"
    list_files.append((pose, ref))

In [13]:
# выбор нужного чекпоинта модели
config = {
    "pretrained_base_model_path": "../Moore-AnimateAnyone/pretrained_weights/stable-diffusion-v1-5/",
    "pretrained_vae_path": "../Moore-AnimateAnyone/pretrained_weights/sd-vae-ft-mse",
    "image_encoder_path": "../Moore-AnimateAnyone/pretrained_weights/image_encoder",
    "denoising_unet_path": "../Moore-AnimateAnyone/exp_output/stage1/denoising_unet-30000.pth",
    "reference_unet_path": "../Moore-AnimateAnyone/exp_output/stage1/reference_unet-30000.pth",
    "pose_guider_path": "../Moore-AnimateAnyone/exp_output/stage1/pose_guider-30000.pth",
    "motion_module_path": "../Moore-AnimateAnyone/exp_output/stage2/motion_module-10000.pth",
    "inference_config": "../Moore-AnimateAnyone/configs/inference/inference_v2.yaml",
    "weight_dtype": "fp16",
    "test_cases":{
        }
    }

In [14]:
def create_yaml_config(list_files, config):
    """
    Содание yaml конфигурационного файла

    Args:
        list_files: (list): список видео и поз для генерации
        config: (dict): словарь с конфишурациями
    """
    test_cases_pattern = '''\n  "./{image_path}":
        - ".{pose_path}"'''
    
    for pose, ref in tqdm(list_files):
        config["test_cases"][ref] = [pose]

    with open(f'data.yaml', 'w') as outfile:
        yaml.dump(config, outfile, default_flow_style=False, sort_keys=False)

In [15]:
create_yaml_config(list_files, config)

100%|██████████| 10/10 [00:00<00:00, 219597.07it/s]


### Инференс

In [16]:
path_config = "data.yaml"

In [19]:
def inference(path_config, save_dir, W=512, H=512, L=90, seed=42, cfg=3.5, steps=20, fps=10):
    """
    Функция для генерации тестовых видео
    """
    config = OmegaConf.load(path_config)
    
    if config.weight_dtype == "fp16":
        weight_dtype = torch.float16
    else:
        weight_dtype = torch.float32
    
    vae = AutoencoderKL.from_pretrained(
        config.pretrained_vae_path,
    ).to("cuda", dtype=weight_dtype)
    
    reference_unet = UNet2DConditionModel.from_pretrained(
        config.pretrained_base_model_path,
        subfolder="unet",
    ).to(dtype=weight_dtype, device="cuda")
    
    inference_config_path = config.inference_config
    infer_config = OmegaConf.load(inference_config_path)
    denoising_unet = UNet3DConditionModel.from_pretrained_2d(
        config.pretrained_base_model_path,
        config.motion_module_path,
        subfolder="unet",
        unet_additional_kwargs=infer_config.unet_additional_kwargs,
    ).to(dtype=weight_dtype, device="cuda")
    
    pose_guider = PoseGuider(320, block_out_channels=(16, 32, 96, 256)).to(
        dtype=weight_dtype, device="cuda"
    )
    
    image_enc = CLIPVisionModelWithProjection.from_pretrained(
        config.image_encoder_path
    ).to(dtype=weight_dtype, device="cuda")
    
    sched_kwargs = OmegaConf.to_container(infer_config.noise_scheduler_kwargs)
    scheduler = DDIMScheduler(**sched_kwargs)
    
    generator = torch.manual_seed(seed)
    
    width, height = W, H
    
    denoising_unet.load_state_dict(
        torch.load(config.denoising_unet_path, map_location="cpu"),
        strict=False,
    )
    reference_unet.load_state_dict(
        torch.load(config.reference_unet_path, map_location="cpu"),
    )
    pose_guider.load_state_dict(
        torch.load(config.pose_guider_path, map_location="cpu"),
    )
    
    pipe = Pose2VideoPipeline(
        vae=vae,
        image_encoder=image_enc,
        reference_unet=reference_unet,
        denoising_unet=denoising_unet,
        pose_guider=pose_guider,
        scheduler=scheduler,
    )
    pipe = pipe.to("cuda", dtype=weight_dtype)
    
    date_str = datetime.now().strftime("%Y%m%d")
    time_str = datetime.now().strftime("%H%M")
    save_dir_name = f"{time_str}--seed_{seed}-{W}x{H}"
    
    for i, ref_image_path in enumerate(config["test_cases"].keys()):
        for pose_video_path in config["test_cases"][ref_image_path]:
            ref_name = Path(ref_image_path).stem
            pose_name = Path(pose_video_path).stem.replace("_kps", "")
    
            ref_image_pil = Image.open(ref_image_path).convert("RGB")
    
            pose_list = []
            pose_tensor_list = []
            pose_images = read_frames(pose_video_path)
            src_fps = get_fps(pose_video_path)
            print(f"pose video has {len(pose_images)} frames, with {src_fps} fps")
            pose_transform = transforms.Compose(
                [transforms.Resize((height, width)), transforms.ToTensor()]
            )
            for pose_image_pil in pose_images[: L]:
                pose_tensor_list.append(pose_transform(pose_image_pil))
                pose_list.append(pose_image_pil)
    
            video = pipe(
                ref_image_pil,
                pose_list,
                width,
                height,
                L,
                steps,
                cfg,
                generator=generator,
            ).videos


            ref_image_tensor = pose_transform(ref_image_pil)
            ref_image_tensor = ref_image_tensor.unsqueeze(1).unsqueeze(0)
            ref_image_tensor = repeat(
                ref_image_tensor, "b c f h w -> b c (repeat f) h w", repeat=L
            )
            pose_tensor = torch.stack(pose_tensor_list, dim=0)
            pose_tensor = pose_tensor.transpose(0, 1)
            pose_tensor = pose_tensor.unsqueeze(0)
            video = torch.cat([ref_image_tensor, pose_tensor, video], dim=0)
    
            save_videos_grid(
                video,
                f"{save_dir}/{ref_name}_{i}.mp4",
                n_rows=3,
                fps=src_fps if fps is None else fps,
            )

In [20]:
save_dir = "test_exaples/"
inference(path_config, save_dir)

Some weights of the model checkpoint were not used when initializing UNet2DConditionModel: 
 ['conv_norm_out.weight, conv_norm_out.bias, conv_out.weight, conv_out.bias']


pose video has 90 frames, with 999/100 fps


100%|██████████| 90/90 [00:03<00:00, 25.34it/s]


pose video has 90 frames, with 999/100 fps


100%|██████████| 90/90 [00:03<00:00, 25.34it/s]


...

100%|██████████| 90/90 [00:03<00:00, 25.34it/s]


pose video has 90 frames, with 999/100 fps


100%|██████████| 90/90 [00:03<00:00, 25.25it/s]


pose video has 90 frames, with 999/100 fps


100%|██████████| 90/90 [00:03<00:00, 25.26it/s]


___